In [1]:
# R3_conformal.ipynb  (revision stage 2b - conformal efficiency, addresses M5)
# Marginal coverage >= 1-alpha holds for ANY model, so it is NOT evidence that the
# second pillar is reproduced. We therefore report EFFICIENCY and CALIBRATION, which
# do discriminate between models, on ONE deployed model per family:
#   logit-WoE   : multivariate logistic on WoE grid
#   stump-boost : multivariate depth-1 gradient boosting on the WoE grid (the SAME
#                 model whose discrimination is reported in stage 3)
# Reported: class-conditional (Mondrian) coverage for good/bad; prediction-set size
# distribution (empty / singleton / both); a realistic cutoff stratum (reject the
# top-decile risk) instead of the 45-55 percentile band; and probability calibration
# (Brier, ECE). PD 'intervals' from binary conformal are dropped (ill-defined, M5);
# we note Venn-Abers as the proper tool for calibrated PD intervals.
# Binning + IV INSIDE training fold (M6). Single-cell; greyscale; dpi600 png+pdf.

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

SEED=42; N_REP=30; ALPHA=0.10; IV_THRESH=0.02
sns.set_theme(style='whitegrid', context='paper'); sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor':'0.2','grid.color':'0.85','font.size':10})
ROOT=os.path.abspath(os.path.join(os.getcwd(),'..')); DATA=os.path.join(ROOT,'data')
TABLES=os.path.join(ROOT,'results','tables'); FIGS=os.path.join(ROOT,'results','figures')
def savefig(fig,name):
    for e in ('png','pdf'): fig.savefig(os.path.join(FIGS,name+'.'+e),dpi=600,bbox_inches='tight')
    plt.close(fig)

dfc=pd.read_parquet(os.path.join(DATA,'gmsc_clean.parquet'))
TARGET='SeriousDlqin2yrs'; FEATURES=[c for c in dfc.columns if c!=TARGET]
y_all=dfc[TARGET].values.astype(int); X_all=dfc[FEATURES]
print('GMSC clean:',dfc.shape,'| nominal coverage %d%%'%int((1-ALPHA)*100))

# ---- in-fold binning (same as R1/R2) ----
def tree_edges(x,yv,max_leaf=6,min_frac=0.05):
    m=x.notna().values; xx=x[m].values.reshape(-1,1); yy=yv[m]
    if len(np.unique(xx))<=2: return []
    t=DecisionTreeClassifier(max_leaf_nodes=max_leaf,min_samples_leaf=int(min_frac*len(xx)),random_state=SEED)
    t.fit(xx,yy); thr=t.tree_.threshold[t.tree_.feature>=0]; return sorted(float(v) for v in thr)
def bin_counts(x,yv,edges,TG,TB,min_n):
    cuts=[-np.inf]+list(edges)+[np.inf]; idx=pd.cut(x,cuts,labels=False,include_lowest=True); recs=[]
    for b in range(len(cuts)-1):
        sel=idx==b; g=int(((yv==0)&sel).sum()); bd=int(((yv==1)&sel).sum())
        if g+bd>0: recs.append({'lo':cuts[b],'hi':cuts[b+1],'good':g,'bad':bd,'miss':False})
    selm=x.isna(); gm=int(((yv==0)&selm).sum()); bm=int(((yv==1)&selm).sum())
    if gm+bm>0: recs.append({'lo':np.nan,'hi':np.nan,'good':gm,'bad':bm,'miss':True})
    for r in recs:
        gr=(r['good']+0.5)/(TG+0.5); br=(r['bad']+0.5)/(TB+0.5); r['woe']=float(np.log(br/gr)); r['n']=r['good']+r['bad']
        if r['miss'] and r['n']<min_n: r['woe']=0.0
    return recs
def enforce_mono(x,yv,edges,TG,TB,min_n):
    e=list(edges)
    while e:
        recs=[r for r in bin_counts(x,yv,e,TG,TB,min_n) if not r['miss']]; br=np.array([r['bad']/r['n'] for r in recs])
        if len(br)<=2 or np.all(np.diff(br)>=-1e-12) or np.all(np.diff(br)<=1e-12): return e
        d=np.diff(br); s=np.sign(np.nansum(d)); e.pop(int(np.argmax(-s*d)))
    return e
def fit_binning(Xtr,ytr):
    TG=int((ytr==0).sum()); TB=int((ytr==1).sum()); min_n=int(0.01*len(ytr)); maps={}; iv={}
    for c in FEATURES:
        recs=bin_counts(Xtr[c],ytr,enforce_mono(Xtr[c],ytr,tree_edges(Xtr[c],ytr),TG,TB,min_n),TG,TB,min_n); maps[c]=recs
        iv[c]=float(np.sum([((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in recs]))
    return maps,[c for c in FEATURES if iv[c]>=IV_THRESH]
def apply_bins(x,recs):
    idx=np.full(len(x),-1,int); fin=[r for r in recs if not r['miss']]
    for p,r in enumerate(fin):
        sel=(~x.isna()).values&(x>r['lo']).values&(x<=r['hi']).values; idx[sel]=p
    mp=[p for p,r in enumerate(recs) if r['miss']]; idx[x.isna().values]=mp[0] if mp else 0; idx[idx==-1]=0; return idx
def woe_mat(Xdf,maps,keep):
    return np.column_stack([np.array([r['woe'] for r in maps[c]])[apply_bins(Xdf[c],maps[c])] for c in keep])

# ---- conformal + calibration helpers ----
def mondrian_qhat(prob_cal,y_cal,alpha):
    q={}
    for cls in (0,1):
        m=y_cal==cls; s=1.0-prob_cal[m][:,cls]; nn=m.sum(); k=min(int(np.ceil((nn+1)*(1-alpha))),nn); q[cls]=np.sort(s)[k-1]
    return q
def sets_mondrian(prob,qh): return prob>=np.array([1-qh[0],1-qh[1]])[None,:]
def brier(p,y): return float(np.mean((p-y)**2))
def ece(p,y,bins=10):
    e=0.0; edges=np.linspace(0,1,bins+1)
    for i in range(bins):
        m=(p>=edges[i])&(p<edges[i+1] if i<bins-1 else p<=edges[i+1])
        if m.sum()>0: e+=abs(p[m].mean()-y[m].mean())*m.sum()
    return float(e/len(p))

rows=[]; size_rows=[]; t0=time.time()
for rep in range(N_REP):
    Xtr,Xtmp,ytr,ytmp=train_test_split(X_all,y_all,test_size=0.4,stratify=y_all,random_state=SEED+rep)
    Xcal,Xte,ycal,yte=train_test_split(Xtmp,ytmp,test_size=0.5,stratify=ytmp,random_state=SEED+rep)
    maps,keep=fit_binning(Xtr,ytr)
    Wtr=woe_mat(Xtr,maps,keep); Wcal=woe_mat(Xcal,maps,keep); Wte=woe_mat(Xte,maps,keep)
    for kind in ('logit','stump'):
        if kind=='logit': m=LogisticRegression(max_iter=1000).fit(Wtr,ytr)
        else: m=GradientBoostingClassifier(max_depth=1,learning_rate=0.3,n_estimators=300,random_state=SEED+rep).fit(Wtr,ytr)
        pc=m.predict_proba(Wcal)[:,1]; pcal=np.column_stack([1-pc,pc])
        pt=m.predict_proba(Wte)[:,1]; ptest=np.column_stack([1-pt,pt])
        qh=mondrian_qhat(pcal,ycal,ALPHA); S=sets_mondrian(ptest,qh)
        cov0=S[yte==0][:,0].mean(); cov1=S[yte==1][:,1].mean()
        sz=S.sum(1); frac_empty=(sz==0).mean(); frac_single=(sz==1).mean(); frac_both=(sz==2).mean()
        # realistic cutoff: reject top-decile risk; coverage among the accepted vs rejected
        thr=np.quantile(pt,0.90); rej=pt>=thr
        cov_rej=S[rej][np.arange(rej.sum()),yte[rej]].mean() if rej.sum()>0 else np.nan
        cov_acc=S[~rej][np.arange((~rej).sum()),yte[~rej]].mean() if (~rej).sum()>0 else np.nan
        rows.append([rep,kind,cov0,cov1,frac_empty,frac_single,frac_both,cov_rej,cov_acc,
                     brier(pt,yte),ece(pt,yte)])
    if (rep+1)%10==0: print('  rep %d/%d (%.0fs)'%(rep+1,N_REP,time.time()-t0))
print('done %.0fs'%(time.time()-t0))

cov=pd.DataFrame(rows,columns=['rep','model','cov_good','cov_bad','empty','single','both',
                               'cov_reject_decile','cov_accept','brier','ece'])
summ=cov.groupby('model').agg(
    cov_good=('cov_good','mean'),cov_bad=('cov_bad','mean'),
    empty=('empty','mean'),single=('single','mean'),both=('both','mean'),
    cov_reject=('cov_reject_decile','mean'),cov_accept=('cov_accept','mean'),
    brier=('brier','mean'),ece=('ece','mean')).round(4)
print('\n=== Conformal EFFICIENCY + CALIBRATION (nominal %.2f), Mondrian ==='%(1-ALPHA))
print(summ.to_string())
print('\nReading: marginal coverage ~0.90 is automatic; what differs is class-conditional')
print('coverage (esp. bad class), set-size mix, decile-cutoff coverage, and calibration.')

cov.round(6).to_csv(os.path.join(TABLES,'R3_conformal_long.csv'),index=False)
summ.to_csv(os.path.join(TABLES,'R3_conformal_summary.csv'))

# ---- FIGURE: set-size mix + class-conditional coverage ----
fig,axes=plt.subplots(1,2,figsize=(11,4.6))
mods=['logit','stump']; x=np.arange(len(mods)); w=0.6
emp=[cov[cov.model==m]['empty'].mean() for m in mods]
sin=[cov[cov.model==m]['single'].mean() for m in mods]
bot=[cov[cov.model==m]['both'].mean() for m in mods]
axes[0].bar(x,emp,w,color='0.3',edgecolor='0.0',label='empty {}')
axes[0].bar(x,sin,w,bottom=emp,color='0.6',edgecolor='0.0',label='singleton')
axes[0].bar(x,bot,w,bottom=np.array(emp)+np.array(sin),color='0.85',edgecolor='0.0',label='both {g,b}')
axes[0].set_xticks(x); axes[0].set_xticklabels(['logistic-WoE','stump-boost'])
axes[0].set_ylabel('prediction-set size mix'); axes[0].set_title('Set-size distribution',fontsize=10)
axes[0].legend(frameon=True,facecolor='white',edgecolor='0.5',fontsize=8)
cg=[cov[cov.model==m]['cov_good'].mean() for m in mods]; cb=[cov[cov.model==m]['cov_bad'].mean() for m in mods]
axes[1].bar(x-0.17,cg,0.34,color='0.45',edgecolor='0.0',label='good')
axes[1].bar(x+0.17,cb,0.34,color='0.7',edgecolor='0.0',label='bad')
axes[1].axhline(1-ALPHA,color='0.0',lw=1.2,ls='--'); axes[1].set_ylim(0.7,1.0)
axes[1].set_xticks(x); axes[1].set_xticklabels(['logistic-WoE','stump-boost'])
axes[1].set_ylabel('class-conditional coverage'); axes[1].set_title('Mondrian coverage by class',fontsize=10)
axes[1].legend(frameon=True,facecolor='white',edgecolor='0.5',fontsize=8)
savefig(fig,'R3_conformal_efficiency')
print('\nSaved: R3_conformal_efficiency, R3_conformal_long, R3_conformal_summary')
print('\n================ R3 HEADLINE ================')
for m in mods:
    s=cov[cov.model==m]
    print('%-6s: cov good=%.3f bad=%.3f | size empty/single/both=%.2f/%.2f/%.2f | Brier=%.4f ECE=%.4f'%(
        m,s['cov_good'].mean(),s['cov_bad'].mean(),s['empty'].mean(),s['single'].mean(),s['both'].mean(),
        s['brier'].mean(),s['ece'].mean()))
print('Efficiency/calibration (not marginal coverage) is the honest second-pillar comparison.')
print('\nR3 complete.')


GMSC clean: (150000, 11) | nominal coverage 90%


  rep 10/30 (54s)


  rep 20/30 (108s)


  rep 30/30 (162s)
done 162s

=== Conformal EFFICIENCY + CALIBRATION (nominal 0.90), Mondrian ===
       cov_good  cov_bad  empty  single    both  cov_reject  cov_accept   brier     ece
model                                                                                  
logit    0.9005   0.9009    0.0  0.6566  0.3434      0.3416      0.9627  0.0516  0.0052
stump    0.9009   0.9011    0.0  0.6578  0.3422      0.3423      0.9630  0.0514  0.0052

Reading: marginal coverage ~0.90 is automatic; what differs is class-conditional
coverage (esp. bad class), set-size mix, decile-cutoff coverage, and calibration.



Saved: R3_conformal_efficiency, R3_conformal_long, R3_conformal_summary

================ R3 HEADLINE ================
logit : cov good=0.900 bad=0.901 | size empty/single/both=0.00/0.66/0.34 | Brier=0.0516 ECE=0.0052
stump : cov good=0.901 bad=0.901 | size empty/single/both=0.00/0.66/0.34 | Brier=0.0514 ECE=0.0052
Efficiency/calibration (not marginal coverage) is the honest second-pillar comparison.

R3 complete.
